# Exercises XP – Week 7, Day 3: BERT
Run in **Google Colab** (a GPU is optional).

## 🌟 Exercise 1: Tokenization with BERT

In [ ]:
!pip install --quiet transformers torch

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

sentence = "Therapists can't replace empathy, but AI tools help with screening."

tokens = tokenizer.tokenize(sentence)
print("Tokens:", tokens)

encoded = tokenizer(sentence, padding="max_length", truncation=True, max_length=24, return_tensors="pt")
ids = encoded["input_ids"][0]
print("\nInput IDs:", ids.tolist())
print("Attention mask:", encoded["attention_mask"][0].tolist())
print("\nTokens with special tokens:", tokenizer.convert_ids_to_tokens(ids))
print("\nSpecial tokens:", {"[CLS]": tokenizer.cls_token_id, "[SEP]": tokenizer.sep_token_id, "[PAD]": tokenizer.pad_token_id})

**What we see:**
- BERT lower-cases the text (`bert-base-uncased`) and splits rare words into **WordPiece** sub-words; continuation pieces start with **`##`** (e.g. `"therapists"` → `"therapist"`, `"##s"`).
- **`[CLS]`** (ID 101) is added at the start. Its final vector is used as the summary of the whole sentence for classification.
- **`[SEP]`** (ID 102) marks the end of a sentence (or separates two sentences).
- **`[PAD]`** (ID 0) fills the sequence up to `max_length`, and the **attention mask** is 1 for real tokens and 0 for padding, so the model ignores the padding.

## 🌟 Exercise 2: Sentiment Analysis with BERT Pipeline

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

sample = "The bootcamp is intense, but I'm learning more than I expected."
result = sentiment(sample)[0]
print(f"Text: {sample}")
print(f"Label: {result['label']} | Confidence: {result['score']:.3f}")

## 🌟 Exercise 3: Building a Custom Sentiment Analyzer

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class BERTSentimentAnalyzer:
    def __init__(self, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.eval()
        self.labels = self.model.config.id2label          # {0: 'NEGATIVE', 1: 'POSITIVE'}

    def preprocess(self, text):
        text = re.sub(r"http\S+", "", text)                # remove links
        text = re.sub(r"\s+", " ", text).strip()           # collapse extra spaces
        return self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128)

    def predict(self, text):
        inputs = self.preprocess(text)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        best = int(torch.argmax(probs))
        return {"text": text, "label": self.labels[best], "confidence": round(float(probs[best]), 3),
                "probabilities": {self.labels[i]: round(float(p), 3) for i, p in enumerate(probs)}}

analyzer = BERTSentimentAnalyzer()
for t in ["I absolutely loved this course!",
          "The instructions were confusing and the deadline was unrealistic.",
          "It was okay, nothing special.",
          "Not bad at all – actually pretty great!   https://example.com"]:
    print(analyzer.predict(t))

Note how *"It was okay, nothing special"* still gets a confident label: this model was trained only on **positive vs. negative** (SST-2), so it has no "neutral" option.

## 🌟 Exercise 4: Understanding BERT for Named Entity Recognition (NER)

In [ ]:
from transformers import AutoModelForTokenClassification

class BERTNamedEntityRecognizer:
    def __init__(self, model_name="dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.eval()
        self.id2label = self.model.config.id2label        # B-PER, I-PER, B-ORG, B-LOC, B-MISC, ..., O

    def recognize(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        predictions = logits.argmax(-1)[0].tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        # Merge word pieces (##) and B-/I- tags into whole entities
        entities, current = [], None
        for token, pred in zip(tokens, predictions):
            label = self.id2label[pred]
            if token in self.tokenizer.all_special_tokens:
                continue
            if token.startswith("##") and current:
                current["text"] += token[2:]
                continue
            if label.startswith("B-") or (label.startswith("I-") and (not current or current["type"] != label[2:])):
                if current:
                    entities.append(current)
                current = {"text": token, "type": label[2:]}
            elif label.startswith("I-") and current:
                current["text"] += " " + token
            else:
                if current:
                    entities.append(current)
                current = None
        if current:
            entities.append(current)
        return tokens, [self.id2label[p] for p in predictions], entities

ner = BERTNamedEntityRecognizer()
text = "Marie Curie worked at the University of Paris and received the Nobel Prize in Stockholm in 1911."
tokens, tags, entities = ner.recognize(text)
for tok, tag in zip(tokens, tags):
    print(f"{tok:15} {tag}")
print("\nEntities found:")
for e in entities:
    print(f"  {e['text']:25} → {e['type']}")

**B-I-O tagging:** **B-** marks the *beginning* of an entity, **I-** a token *inside* the same entity, and **O** a token *outside* any entity. Types: PER = person, ORG = organisation, LOC = location, MISC = other (e.g. products or events).

## 🌟 Exercise 5: Comparing BERT and GPT

| | **BERT** | **GPT** |
|---|---|---|
| **Architecture** | Transformer **encoder** only; **bidirectional** (each token sees left and right context) | Transformer **decoder** only; **unidirectional / causal** (each token sees only previous tokens) |
| **Pre-training** | Masked Language Modelling (+ Next Sentence Prediction in the original) | Next-token prediction (causal language modelling) |
| **Primary purpose** | **Understanding** text | **Generating** text |
| **Common use cases** | Classification, sentiment, NER, question answering (extracting answers), search and **embeddings for retrieval** | Chatbots, writing assistance, summarisation, code generation, translation via prompting |
| **Strengths** | Deep contextual understanding of the whole input; smaller and efficient; excellent after fine-tuning on small labelled datasets | Fluent, coherent generation; very flexible (zero- and few-shot via prompts); scales to very large, capable models |
| **Weaknesses** | Can't naturally generate long text; needs fine-tuning for each task; limited input length (512 tokens) | Only left-to-right context; large models are expensive; can produce confident but wrong text (hallucinations) |

**Reflection:** both are built from the same Transformer blocks with self-attention and are pre-trained on large unlabelled text, then adapted. The key difference is the **direction of attention and the training objective**: BERT sees the whole sentence at once and learns to *fill in* blanks, which makes it a strong *reader*; GPT predicts what comes *next*, which makes it a strong *writer*. They're often used **together**, as in RAG systems below: BERT-style models find relevant information, and GPT-style models write the answer.

## 🌟 Exercise 6: Exploring BERT Applications in Retrieval-Augmented Generation (RAG)

**What is RAG?** A generative model like GPT only knows what was in its training data, which may be outdated, and it can invent facts. **Retrieval-Augmented Generation** adds a search step: before answering, the system **retrieves** relevant passages from an external knowledge base (company documents, medical guidelines, a website) and gives them to the generator as context, so the answer is **grounded in real sources**.

**BERT's role in the retrieval component:** BERT-style encoders (e.g. Sentence-BERT, DPR, or modern embedding models) turn text into **dense vectors (embeddings)** that capture meaning, so two texts about the same idea end up close together even if they use different words ("heart attack" ≈ "myocardial infarction").

**How BERT creates embeddings for documents and queries:**
1. Documents are split into chunks (e.g. paragraphs).
2. Each chunk is passed through the encoder, and its token vectors are combined, often by **mean pooling** or using the **[CLS]** vector, into one fixed-size embedding (e.g. 384 or 768 numbers).
3. At question time, the user's query is encoded **the same way** into a query embedding.

**How a vector database matches queries to documents:** all document embeddings are stored in a **vector database** (FAISS, Pinecone, Chroma, Weaviate…) with an index built for fast **nearest-neighbour search**. The query embedding is compared with the stored vectors using **cosine similarity** (or dot product), and the top-k most similar chunks are returned, often in milliseconds, even across millions of documents.

**Example – a clinic's patient-information assistant:**
1. A patient asks: *"Can I take ibuprofen after my knee surgery?"*
2. A BERT-based encoder turns the question into a vector, and the vector database returns the 3 most similar passages from the clinic's post-surgery care guidelines.
3. These passages are inserted into the prompt: *"Using only the information below, answer the patient's question… [passages]"*.
4. A GPT-style model writes a clear, friendly answer based on those guidelines, citing the source and advising the patient to confirm with their surgeon.

This way BERT provides **accurate, up-to-date retrieval**, and GPT provides **fluent generation**, which reduces hallucinations and lets the system use knowledge the generator was never trained on.